<a href="https://colab.research.google.com/github/NicolasRodrigues07/02PY_CP_1CCPW/blob/main/Chatbot_GoodWe_Sprint2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GoodWe EV Chatbot — LLaMA 3.2 — ChargeGrid Intelligence

Projeto Sprint 2 — Desenvolvimento e Entrega do Chatbot GoodWe

**Técnicas utilizadas:** RAG (Retrieval-Augmented Generation), few-shot prompting, memória de histórico

**Modelo:** `meta-llama/Llama-3.2-1B-Instruct` via HuggingFace  
**Framework:** LangChain + LangGraph  
**API Key:** gerenciada via Google Colab Secrets (`HF_TOKEN`)

In [23]:
%pip install --quiet google-generativeai langchain langchain-community langchain-core pypdf langchain-text-splitters langgraph sentence-transformers

In [25]:
import os
from google.colab import userdata, files
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langgraph.graph import START, StateGraph
from typing_extensions import List, TypedDict
from langchain_huggingface import HuggingFaceEmbeddings
import google.generativeai as genai

genai.configure(api_key=userdata.get('GEMINI_API_KEY'))
print('Login Gemini OK')

Login Gemini OK


In [26]:
model = genai.GenerativeModel(
    model_name='gemma-3-4b-it',
    generation_config=genai.GenerationConfig(
        max_output_tokens=512,
        temperature=0.3,
    )
)

def chamar_modelo(messages):
    # monta o histórico no formato do Gemini
    history = []
    for msg in messages[:-1]:
        role = 'user' if msg['role'] == 'user' else 'model'
        history.append({'role': role, 'parts': [msg['content']]})

    chat = model.start_chat(history=history)
    response = chat.send_message(messages[-1]['content'])
    return response.text

print('Modelo carregado!')

Modelo carregado!


## Upload dos PDFs

Faça upload dos PDFs do projeto GoodWe (ex: `chargegrid.pdf`, manuais técnicos, etc.).

In [ ]:
uploaded = files.upload()
pdf_files = [f for f in uploaded.keys() if f.endswith('.pdf')]
print(f'PDFs carregados: {pdf_files}')

In [ ]:
# Embeddings + vector store
embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
vector_store = InMemoryVectorStore(embeddings)
print('Vector store pronto!')

In [ ]:
# Carregando e indexando os PDFs
all_docs = []
for pdf in pdf_files:
    loader = PyPDFLoader(pdf)
    docs = loader.load()
    all_docs.extend(docs)
    print(f'{pdf}: {len(docs)} paginas')

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
all_splits = text_splitter.split_documents(all_docs)
print(f'Split em {len(all_splits)} chunks.')

document_ids = vector_store.add_documents(documents=all_splits)
print(f'Indexados {len(document_ids)} chunks. Pronto!')

## RAG + System Prompt + Few-Shot

O system prompt define o escopo do chatbot (ChargeGrid Intelligence / EV ChargeOps) e inclui exemplos few-shot para guiar o estilo das respostas.

In [ ]:
FEW_SHOT_EXAMPLES = """
Exemplos de como responder:

Pergunta: O que é a ChargeGrid?
Resposta: A ChargeGrid é uma rede de carregadores de veículos elétricos interconectados que gerenciam a distribuição de energia de forma inteligente, evitando sobrecargas na rede elétrica.

Pergunta: Qual o protocolo de comunicação usado?
Resposta: O GoodWe HCA G2 utiliza o protocolo Modbus/LAN para comunicação com os sistemas de gestão da ChargeGrid.

Pergunta: O carregador funciona no Brasil?
Resposta: Sim. Os modelos da linha HCA G2 são compatíveis com redes de 220/380 Vac, atendendo perfeitamente o padrão elétrico brasileiro.
"""


class State(TypedDict):
    question: str
    context: List[Document]
    answer: str
    history_list: list


def retrieve(state: State):
    retrieved_docs = vector_store.similarity_search(state['question'])
    return {'context': retrieved_docs}


def generate(state: State):
    docs_content = '\n\n'.join(doc.page_content for doc in state['context'])

    system_content = (
        "Você é o assistente oficial da GoodWe para o EV Challenge 2026, especializado em "
        "ChargeGrid Intelligence e EV ChargeOps. "
        "Responda APENAS com base no contexto dos documentos fornecidos. "
        "Se a resposta não estiver no contexto, diga: 'Não encontrei essa informação nos documentos do projeto.' "
        "Seja direto, objetivo e responda sempre em português.\n\n"
        f"{FEW_SHOT_EXAMPLES}\n"
        f"Contexto dos documentos:\n{docs_content}"
    )

    messages = []
    history = state.get('history_list', [])

    if not history:
        messages.append({"role": "user", "content": f"{system_content}\n\nPergunta: {state['question']}"})
    else:
        primeira_pergunta, primeira_resposta = history[0]
        messages.append({"role": "user", "content": f"{system_content}\n\nPergunta: {primeira_pergunta}"})
        messages.append({"role": "model", "content": primeira_resposta})

        for pergunta, resposta in history[1:]:
            messages.append({"role": "user", "content": pergunta})
            messages.append({"role": "model", "content": resposta})

        messages.append({"role": "user", "content": state['question']})

    answer = chamar_modelo(messages)
    return {'answer': answer}


graph_builder = StateGraph(State).add_sequence([retrieve, generate])
graph_builder.add_edge(START, 'retrieve')
graph = graph_builder.compile()

print('RAG pronto!')

## Testes — Modelo de Avaliação Sprint 2

Executando os 5 casos de teste definidos na Sprint 1 e registrando os resultados.

In [ ]:
casos_de_teste = [
    "O que é a ChargeGrid Intelligence e qual problema ela resolve?",
    "Quais são as especificações elétricas do GoodWe HCA G2 para o mercado brasileiro?",
    "Como funciona o gerenciamento dinâmico de carga (DLM) na ChargeGrid?",
    "Quais protocolos de comunicação o carregador GoodWe utiliza para integração com a smart grid?",
    "O que acontece com a distribuição de energia quando vários carros são conectados ao mesmo tempo?"
]

print('=' * 60)
print('RESULTADOS DOS TESTES — SPRINT 2')
print('=' * 60)

for i, pergunta in enumerate(casos_de_teste, 1):
    print(f'\n[TESTE {i}]')
    print(f'Pergunta: {pergunta}')

    result = graph.invoke({
        'question': pergunta,
        'history_list': [],
        'context': [],
        'answer': ''
    })

    print(f'Resposta: {result["answer"]}')
    print(f'Chunks recuperados: {len(result["context"])}')
    print('-' * 60)

## Chatbot Interativo com Histórico

Digite `sair` para encerrar.  
Digite `/historico` para ver o histórico da conversa.  
Digite `/limpar` para apagar o histórico.

In [ ]:
print('GoodWe Chatbot — EV Challenge 2026')
print('Digite sair para encerrar')
print('Digite /historico para ver o historico da conversa')
print('Digite /limpar para apagar o historico')

historico = []

while True:
    pergunta = input('\nVoce: ').strip()

    if pergunta.lower() in ['sair', 'exit']:
        print('Ate logo!')
        break

    if pergunta.lower() == '/historico':
        print('\n--- Historico da conversa ---')
        if not historico:
            print('(vazio)')
        else:
            for i, (p, r) in enumerate(historico, 1):
                print(f'[{i}] Voce: {p}')
                print(f'[{i}] Bot: {r}')
        continue

    if pergunta.lower() == '/limpar':
        historico = []
        print('Historico apagado!')
        continue

    if not pergunta:
        continue

    result = graph.invoke({
        'question': pergunta,
        'history_list': historico,
        'context': [],
        'answer': ''
    })

    resposta = result['answer']
    historico.append((pergunta, resposta))

    print(f'\nBot: {resposta}')
    print(f'(baseado em {len(result["context"])} trechos dos PDFs | {len(historico)} mensagem(ns) no historico)')